# Day 4: PHI Governance & De-Identification Engine

This notebook provides an interactive de-id demo comparing raw vs. masked synthetic patient dataframes.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, date_add, when
from pyspark.sql.types import StringType, DateType
import hashlib
import datetime

spark = SparkSession.builder.appName("Day04_Deid_Demo").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# Define De-ID UDFs
import os
def fpe_encrypt(value, secret_key=None):
    if secret_key is None:
        secret_key = os.environ.get("HIPAA_SECRET_KEY", "")
    if not value: return None
    hasher = hashlib.sha256()
    hasher.update(f"{value}_{secret_key}".encode('utf-8'))
    return hasher.hexdigest()[:len(str(value))]

fpe_encrypt_udf = udf(fpe_encrypt, StringType())

def get_date_shift_offset(patient_id, secret_key=None):
    if secret_key is None:
        secret_key = os.environ.get("HIPAA_SECRET_KEY", "")
    if not patient_id: return 0
    hasher = hashlib.md5()
    hasher.update(f"{patient_id}_{secret_key}".encode('utf-8'))
    hash_int = int(hasher.hexdigest(), 16)
    return (hash_int % 61) - 30

get_date_shift_offset_udf = udf(get_date_shift_offset, StringType())

def redact_free_text(text):
    if not text: return None
    return str(text).replace("John", "[REDACTED_NAME]").replace("Doe", "[REDACTED_NAME]")

redact_free_text_udf = udf(redact_free_text, StringType())

In [ ]:
# Create Synthetic Raw DataFrame
data = [
    ("P1001", "123-45-6789", "MRN991", datetime.date(1980, 5, 15), "Patient John Doe reported a headache."),
    ("P1002", "987-65-4321", "MRN992", datetime.date(1992, 11, 2), "Follow up visit for Jane Smith."),
    ("P1003", "555-66-7777", "MRN993", datetime.date(1975, 8, 20), "Dr. ordered tests for John.")
]
schema = ["patient_id", "ssn", "mrn", "birth_date", "clinical_notes"]
df_raw = spark.createDataFrame(data, schema)

print("RAW DATA:")
df_raw.show(truncate=False)

In [ ]:
# Apply De-Identification
df_deid = df_raw \
    .withColumn("date_shift_offset", get_date_shift_offset_udf(col("patient_id")).cast("int")) \
    .withColumn("birth_date", date_add(col("birth_date"), col("date_shift_offset"))) \
    .withColumn("ssn", fpe_encrypt_udf(col("ssn"))) \
    .withColumn("mrn", fpe_encrypt_udf(col("mrn"))) \
    .withColumn("clinical_notes", redact_free_text_udf(col("clinical_notes"))) \
    .drop("date_shift_offset")

print("DE-IDENTIFIED DATA:")
df_deid.show(truncate=False)